# NutriEpiDB: A Database For Dietary Compounds With Epigenetic Targets Exploration with `mlcroissant`
This notebook demonstrates loading, overview, extraction, processing, and visualization for a Croissant-based dataset using the `mlcroissant` library.

### Dataset Source
NutriEpiDB is provided by FAIR^2 at [https://sen.science/doi/10.71728/senscience.sx3s-9110/fair2.json](https://sen.science/doi/10.71728/senscience.sx3s-9110/fair2.json), featuring curated compound–epigenetic target associations from primary literature.

In [ ]:
# Ensure the latest mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the NutriEpiDB dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.sx3s-9110/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print summary metadata
print(f"{metadata.name}: {metadata.description}\n\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets and their structure. All references use `@id` fields.

List record set IDs, their field IDs, and main columns.

In [ ]:
# List all record sets by their @id
record_sets = [rs['@id'] for rs in dataset.metadata.recordSet]
print("Available record sets (@id):")
for rs in record_sets:
    print(f"- {rs}")

# For each record set, print its fields and columns
for rs in dataset.metadata.recordSet:
    print(f"\nRecord Set @id: {rs['@id']}")
    field_ids = []
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            field_ids.append(f['@id'])
    print(f"Fields (@id): {field_ids}")
    # Print columns if present
    if 'column' in rs:
        columns = rs['column']
        if isinstance(columns, dict):
            columns = [columns]
        column_ids = [c['@id'] for c in columns]
        print(f"Columns (@id): {column_ids}")
    else:
        print("No columns listed.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id`s discovered above.

In [ ]:
# Extract record set dataframes by their @id
dataframes = {}
print("\nLoading data for each record set...")
for rs in record_sets:
    records = list(dataset.records(record_set=rs))
    df = pd.DataFrame(records)
    dataframes[rs] = df
    print(f"Record set '{rs}' loaded with shape {df.shape}.")

# Display column names for one record set as example
if record_sets:
    selected_rs = record_sets[0]
    print(f"\nColumns in record set {selected_rs}:\n{dataframes[selected_rs].columns.tolist()}")
    display(dataframes[selected_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps for a record set, such as filtering records, normalizing numeric fields, and grouping.

In [ ]:
# Choose a record set and numeric field by @id
selected_rs = record_sets[0]  # Use first record set
df = dataframes[selected_rs]

# Find a numeric field (try common ones)
numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or df[col].apply(lambda x: isinstance(x, (int, float))).all()]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = df.columns[0]  # fallback

# Filter based on threshold
threshold = 10
try:
    filtered_df = df[df[numeric_field_id] > threshold]
except Exception:
    filtered_df = df.copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize field
try:
    field_mean = filtered_df[numeric_field_id].mean()
    field_std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - field_mean) / field_std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
except Exception:
    print(f"Could not normalize {numeric_field_id}, non-numeric or missing values.")

# Try grouping by another available field
group_field_id = None
for gcol in df.columns:
    if df[gcol].dtype == 'object' and gcol != numeric_field_id and df[gcol].nunique() < 20:
        group_field_id = gcol
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution for the selected numeric field and relationship with group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot for numeric field grouped by group_field_id
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We successfully loaded NutriEpiDB using Croissant, extracted and explored structured record sets via their `@id`, processed and visualized a numeric field, and grouped by key attributes. This approach enables reproducible and FAIR data workflows for computational modeling and exploratory analysis in nutrigenomics and epigenetic target research.